# Exploring the TensorFlow Ecosystem: A Comprehensive Overview

Maintained runnable companion. Original source code is preserved verbatim in the nonexecuting appendix. Git notebooks are output-free. No API key or paid inference is needed.

In [ ]:
import sys, tempfile, json, math, socket, subprocess, time
from pathlib import Path
from importlib.metadata import version
import urllib.request
import numpy as np
import tensorflow as tf
tf.keras.utils.set_random_seed(42)
tf.config.threading.set_inter_op_parallelism_threads(2)
tf.config.threading.set_intra_op_parallelism_threads(4)
work_dir = Path(tempfile.mkdtemp(prefix="tdb-tensorflow-"))
print("Python:", sys.version.split()[0])
print("TensorFlow:", tf.__version__, "Keras:", version("keras"), "TensorBoard:", version("tensorboard"))
print("Runtime artifacts are written to a new temporary directory.")

In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train = x_train.reshape(-1, 784).astype("float32") / 255.0
x_test = x_test.reshape(-1, 784).astype("float32") / 255.0
options = tf.data.Options()
options.threading.private_threadpool_size = 1
train_dataset = tf.data.Dataset.from_tensor_slices((x_train, y_train)).batch(32).with_options(options)
test_dataset = tf.data.Dataset.from_tensor_slices((x_test, y_test)).batch(32).with_options(options)
assert x_train.shape == (60000, 784) and x_test.shape == (10000, 784)
print("MNIST training:", x_train.shape, "test:", x_test.shape)
print("Labels:", sorted(np.unique(y_train).tolist()))

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(784,)),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dense(10, activation="softmax")])
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
log_dir = work_dir / "logs_keras"
callbacks = [tf.keras.callbacks.TensorBoard(str(log_dir))]
history = model.fit(train_dataset, epochs=5, verbose=0,
                    validation_data=test_dataset, callbacks=callbacks)
for epoch, (loss, accuracy, val_loss, val_accuracy) in enumerate(zip(
        history.history["loss"], history.history["accuracy"],
        history.history["val_loss"], history.history["val_accuracy"]), 1):
    print(f"Epoch {epoch}: loss={loss:.6f}, accuracy={accuracy:.6f}, val_loss={val_loss:.6f}, val_accuracy={val_accuracy:.6f}")
assert int(model.optimizer.iterations) == 5 * math.ceil(60000 / 32)

In [ ]:
def free_local_port():
    with socket.socket() as sock:
        sock.bind(("127.0.0.1", 0))
        return sock.getsockname()[1]
port = free_local_port()
command = [sys.executable, "-m", "tensorboard.main", "--logdir", str(log_dir),
           "--host", "127.0.0.1", "--port", str(port)]
with (work_dir / "tensorboard-server.log").open("w") as server_log:
    server = subprocess.Popen(command, stdout=server_log, stderr=server_log)
    try:
        for _ in range(100):
            if server.poll() is not None:
                raise RuntimeError("TensorBoard exited before readiness")
            try:
                with urllib.request.urlopen(f"http://127.0.0.1:{port}/", timeout=2) as response:
                    page = response.read().decode()
                    assert response.status == 200 and "tensorboard" in page.lower()
                    break
            except OSError:
                time.sleep(0.2)
        else:
            raise TimeoutError("TensorBoard did not become ready")
        print("Real TensorBoard loopback HTTP: 200; dashboard served.")
    finally:
        server.terminate()
        server.wait(timeout=15)

In [ ]:
writer = tf.summary.create_file_writer(str(work_dir / "model_logs"))
with writer.as_default():
    tf.summary.scalar("custom_log", 10, step=3)
writer.flush()
print("custom_log=10 logged at step 3.")

In [ ]:
accuracy = tf.keras.metrics.Accuracy()
ground_truth, predictions = [1, 0, 1], [1, 0, 0]
accuracy.update_state(ground_truth, predictions)
with writer.as_default():
    tf.summary.scalar("accuracy", accuracy.result(), step=4)
writer.flush()
print("Accuracy:", float(accuracy.result()))
print("accuracy logged at step 4.")

In [ ]:
swish_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(784,)),
    tf.keras.layers.Dense(256),
    tf.keras.layers.Activation("swish"),
    tf.keras.layers.Dense(10, activation="softmax")])
swish_model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
swish_history = swish_model.fit(train_dataset, epochs=10, verbose=0)
for epoch, (loss, accuracy_value) in enumerate(zip(swish_history.history["loss"], swish_history.history["accuracy"]), 1):
    print(f"Epoch {epoch}: loss={loss:.6f}, accuracy={accuracy_value:.6f}")
assert int(swish_model.optimizer.iterations) == 10 * math.ceil(60000 / 32)
heldout = swish_model.evaluate(test_dataset, verbose=0, return_dict=True)
print("Held-out MNIST:", json.dumps({k: round(float(v), 6) for k, v in heldout.items()}))

In [ ]:
lite_model = tf.keras.applications.MobileNetV2(weights="imagenet", input_shape=(224, 224, 3))
export_dir = work_dir / "mobilenet_saved_model"
lite_model.export(str(export_dir), verbose=False)
converter = tf.lite.TFLiteConverter.from_saved_model(str(export_dir))
tflite_model = converter.convert()
lite_path = work_dir / "model.tflite"
lite_path.write_bytes(tflite_model)
assert tflite_model[4:8] == b"TFL3"
print("Model successfully converted to TensorFlow Lite format.")
print("FlatBuffer bytes:", len(tflite_model))

In [ ]:
interpreter = tf.lite.Interpreter(model_path=str(lite_path), num_threads=2)
interpreter.allocate_tensors()
input_detail = interpreter.get_input_details()[0]
output_detail = interpreter.get_output_details()[0]
test_input = np.random.default_rng(42).uniform(-1, 1, (1, 224, 224, 3)).astype(np.float32)
keras_prediction = lite_model(test_input, training=False).numpy()
interpreter.set_tensor(input_detail["index"], test_input)
interpreter.invoke()
lite_prediction = interpreter.get_tensor(output_detail["index"])
max_error = float(np.max(np.abs(keras_prediction - lite_prediction)))
np.testing.assert_allclose(lite_prediction, keras_prediction, rtol=1e-3, atol=1e-4)
assert lite_prediction.shape == (1, 1000) and np.isfinite(lite_prediction).all()
np.testing.assert_allclose(lite_prediction.sum(axis=-1), 1, atol=1e-5)
print("Keras and Lite output shape:", lite_prediction.shape)
print("Maximum absolute probability difference:", f"{max_error:.8f}")
print("Top-class indices match:", bool(lite_prediction.argmax() == keras_prediction.argmax()))

In [ ]:
javascript_source = "import * as tf from '@tensorflow/tfjs';\n\nasync function main() {\n  let initialized = false;\n  if (typeof window !== 'undefined') {\n    try { initialized = await tf.setBackend('webgl'); }\n    catch { initialized = false; }\n  }\n  if (!initialized) await tf.setBackend('cpu');\n  await tf.ready();\n  console.log('Backend:', tf.getBackend());\n\n  const model = tf.sequential();\n  model.add(tf.layers.dense({units: 64, activation: 'relu', inputShape: [4],\n    kernelInitializer: tf.initializers.glorotUniform({seed: 42})}));\n  model.add(tf.layers.dense({units: 3, activation: 'softmax',\n    kernelInitializer: tf.initializers.glorotUniform({seed: 43})}));\n  model.compile({optimizer: 'sgd', loss: 'categoricalCrossentropy', metrics: ['accuracy']});\n\n  const data = tf.randomNormal([100, 4], 0, 1, 'float32', 42);\n  const classIds = tf.randomUniform([100], 0, 3, 'int32', 43);\n  const labels = tf.oneHot(classIds, 3);\n  const labelSums = labels.sum(1);\n  if (!(await labelSums.data()).every(value => value === 1)) throw Error('Invalid class labels');\n  labelSums.dispose();\n  classIds.dispose();\n  let epochs = 0;\n  const history = await model.fit(data, labels, {\n    epochs: 10, shuffle: false,\n    callbacks: {onEpochEnd(epoch, logs) {\n      if (!Number.isFinite(logs.loss) || !Number.isFinite(logs.acc)) throw Error('Non-finite training');\n      epochs++;\n      console.log(`Epoch ${epoch + 1}: loss=${logs.loss.toFixed(6)}, accuracy=${logs.acc.toFixed(6)}`);\n    }}\n  });\n  if (epochs !== 10 || history.history.loss.length !== 10) throw Error('Incomplete training');\n  const prediction = model.predict(data);\n  if (prediction.shape.join(',') !== '100,3' || !(await prediction.data()).every(Number.isFinite)) {\n    throw Error('Invalid prediction shape or values');\n  }\n  const probabilities = await prediction.sum(1).data();\n  if (!probabilities.every(value => Math.abs(value - 1) < 1e-5)) throw Error('Invalid probabilities');\n  console.log('PASS: all 100 rows, valid one-hot labels, ten complete epochs, finite 100x3 predictions and normalized probabilities. Random labels are not a quality benchmark.');\n  prediction.dispose();\n  model.dispose();\n  data.dispose();\n  labels.dispose();\n  return {backend: tf.getBackend(), epochs, rows: 100, classes: 3};\n}\n\nmain().then(result => {\n  console.log(JSON.stringify(result));\n  if (typeof document !== 'undefined') {\n    document.querySelector('#result').textContent = JSON.stringify(result, null, 2);\n  }\n}).catch(error => {\n  console.error(error);\n  if (typeof document !== 'undefined') document.querySelector('#result').textContent = 'ERROR: ' + error.message;\n  else process.exitCode = 1;\n});\n"
js_dir = work_dir / "tfjs"
js_dir.mkdir()
(js_dir / "example.mjs").write_text(javascript_source)
installation = subprocess.run(["npm", "install", "--prefix", str(js_dir), "--no-audit", "--no-fund", "@tensorflow/tfjs@4.22.0"], check=True, capture_output=True, text=True)
result = subprocess.run(["node", str(js_dir / "example.mjs")], check=True, capture_output=True, text=True)
if result.stderr:
    print(result.stderr, file=sys.stderr)
print(result.stdout.strip())
js_result = json.loads(result.stdout.strip().splitlines()[-1])
assert js_result == {"backend": "cpu", "epochs": 10, "rows": 100, "classes": 3}


In [ ]:
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator
custom_events = EventAccumulator(str(work_dir / "model_logs")).Reload()
assert set(custom_events.Tags()["tensors"]) == {"custom_log", "accuracy"}
custom = custom_events.Tensors("custom_log")
measured = custom_events.Tensors("accuracy")
assert len(custom) == len(measured) == 1 and custom[0].step == 3 and measured[0].step == 4
assert float(tf.make_ndarray(custom[0].tensor_proto)) == 10
np.testing.assert_allclose(float(tf.make_ndarray(measured[0].tensor_proto)), 2/3, atol=1e-7)
events = EventAccumulator(str(log_dir / "train")).Reload()
logged_losses = events.Tensors("epoch_loss")
assert len(logged_losses) == 5
np.testing.assert_allclose([float(tf.make_ndarray(e.tensor_proto)) for e in logged_losses], history.history["loss"], atol=1e-7)
probe = tf.constant([-5., -1., 0., 1., 5.])
np.testing.assert_allclose(tf.keras.activations.swish(probe), probe * tf.math.sigmoid(probe), atol=1e-7)
for fitted in [history, swish_history]:
    assert all(np.isfinite(values).all() for values in fitted.history.values())
print("PASS: full 60000/10000 MNIST rows, all 5+10 epochs/28125 optimizer updates; actual TensorBoard events and HTTP; exact 2/3 accuracy logging; Swish equation; real ImageNet MobileNet conversion/inference parity.")

## Original source code — nonexecuting appendix

### Original block 1

```text
callbacks = [tf.keras.callbacks.TensorBoard('./logs_keras')]
model.fit(x_train, y_train, epochs=5, verbose=1, validation_data=(x_test,
y_test), callbacks=callbacks)
```

### Original block 2

```text
$ tensorboard --logdir ./logs_keras
```

### Original block 3

```text
writer = tf.summary.create_file_writer('./model_logs')
with writer.as_default():
  tf.summary.scalar('custom_log', 10, step=3)
```

### Original block 4

```text
accuracy = tf.keras.metrics.Accuracy()
ground_truth, predictions = [1, 0, 1], [1, 0, 0] # in practice this would
come from the model
accuracy.update_state(ground_truth, predictions)
tf.summary.scalar('accuracy', accuracy.result(), step=4)
```

### Original block 5

```text
import tensorflow as tf
import tensorflow_addons as tfa

# Create a simple neural network model
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(784,)),
    tf.keras.layers.Dense(256),
    tfa.layers.Swish(),  # Using Swish activation from TensorFlow Addons
    tf.keras.layers.Dense(10, activation='softmax')
])

# Compile the model
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Train the model on a dataset
# (Code for loading and preparing the dataset is not shown in this example)
model.fit(train_dataset, epochs=10)
```

### Original block 6

```text
import tensorflow as tf

# Load a pre-trained TensorFlow model
model = tf.keras.applications.MobileNetV2(weights='imagenet', input_shape=(224, 224, 3))

# Convert the TensorFlow model to TensorFlow Lite format
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

# Save the converted model to a .tflite file
with open('model.tflite', 'wb') as f:
    f.write(tflite_model)

print("Model successfully converted to TensorFlow Lite format.")
```

### Original block 7

```text
// Import TensorFlow.js
import * as tf from '@tensorflow/tfjs';

// Check if GPU acceleration is available
if (tf.ENV.get('WEBGL_VERSION') === 2 && tf.ENV.get('WEBGL_GPU_NUM_GROUPS') > 0) {
  console.log('GPU acceleration is available!');
  tf.setBackend('webgl');
} else {
  console.log('GPU acceleration is not available; using the CPU.');
  tf.setBackend('cpu');
}

// Create a simple neural network model
const model = tf.sequential();
model.add(tf.layers.dense({ units: 64, activation: 'relu', inputShape: [4] }));
model.add(tf.layers.dense({ units: 3, activation: 'softmax' }));

// Compile the model
model.compile({
  optimizer: 'sgd',
  loss: 'categoricalCrossentropy',
  metrics: ['accuracy'],
});

// Generate some example data
const data = tf.random.normal([100, 4]);
const labels = tf.random.uniform([100, 3], 0, 3, 'int32');

// Train the model with GPU acceleration
model.fit(data, labels, {
  epochs: 10,
  callbacks: tf.node.tensorBoard('/tmp/tfjs_logs'),
}).then((info) => {
  console.log('Training completed:', info);
});
```